# Simulation de trafic (Python) — Explication du code

## Objectif

Ce notebook explique uniquement la partie **Python** du projet : les fonctions et méthodes utilisées pour générer du trafic réseau contrôlé (ICMP, HTTP, tentatives SSH, scan TCP).

Le framework d’interface (ex. Flask) **n’est pas expliqué ici**. L’objectif est de comprendre le code Python et son comportement.

## 1. Bibliothèques Python utilisées

Ce code s’appuie sur des bibliothèques Python standards et des bibliothèques orientées réseau :

- `subprocess` : exécuter des commandes système (ex. `ping`) et récupérer la sortie.
- `time` : ajouter des délais entre les requêtes afin de contrôler l’intensité.
- `requests` : envoyer des requêtes HTTP vers le serveur web (Nginx).
- `paramiko` : initier des connexions SSH et générer des échecs d’authentification.
- `socket` : tester des ports TCP (scan léger) via `connect_ex`.

Ces appels génèrent du trafic et des traces observables dans Snort (réseau) et dans Wazuh via les logs de la victime (auth/logs web).

## 2. Importation des modules

In [ ]:
import subprocess
import time
import socket
import requests
import paramiko

## 3. Fonction utilitaire : exécuter une commande système

### Rôle
Cette fonction exécute une commande (liste d’arguments), capture la sortie standard et la sortie d’erreur, puis retourne le code de retour.

### Intérêt dans le projet
Elle est utile pour exécuter un `ping` ICMP et récupérer directement le résultat.

In [ ]:
def run_cmd(cmd: list[str]) -> int:
    """Exécute une commande système et affiche stdout/stderr."""
    p = subprocess.run(cmd, check=False , capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.strip())
    if p.stderr:
        print(p.stderr.strip())
    return p.returncode

## 4. Simulation ICMP : ping burst

### Rôle
Générer un trafic ICMP (ping) vers la machine victime.

### Détection attendue
- Snort : alerte ICMP (règle ICMP ping)

### Paramètres
- `host` : IP victime
- `count` : nombre de paquets ICMP
- `interval` : intervalle entre paquets (contrôle de l’intensité)

In [ ]:
def icmp_ping_burst(host: str, count: int = 10, interval: float = 0.2) -> int:
    return run_cmd(["ping", "-c", str(count), "-i", str(interval), host])

## 5. Simulation HTTP : requêtes GET répétées

### Rôle
Envoyer des requêtes HTTP vers le serveur Nginx de la victime. Cela crée :
- du trafic réseau TCP/80 (Snort)
- des logs applicatifs côté Nginx (`/var/log/nginx/access.log`) (Wazuh)

### Paramètres
- `url` : URL de la victime (ex. `http://10.10.10.129/`)
- `repeat` : nombre de requêtes
- `delay` : délai entre requêtes
- `timeout` : timeout réseau
- `user_agent` : identifiant dans le header HTTP (utile pour tracer dans les logs)

In [ ]:
def http_burst(url: str, repeat: int = 30, delay: float = 0.05, timeout: float = 1.0,
              user_agent: str = "SOC-LAB-TEST/1.0") -> list[str]:
    out = []
    headers = {"User-Agent": user_agent}
    for i in range(repeat):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            out.append(f"[{i+1}/{repeat}] HTTP {r.status_code} ({len(r.content)} bytes)")
        except Exception as e:
            out.append(f"[{i+1}/{repeat}] ERROR: {type(e).__name__}: {e}")
        time.sleep(delay)
    return out

## 6. Simulation SSH : échecs d’authentification

### Rôle
Déclencher des tentatives SSH avec identifiants invalides. Objectif : générer des logs dans `/var/log/auth.log` sur la victime.

### Détection attendue
- Wazuh : événements `sshd` (échec d’authentification)
- Snort : alerte SSH (selon votre règle et seuil)

### Remarque
Le code ci-dessous génère des tentatives contrôlées. Il ne réalise pas d’exploitation.

In [ ]:
def ssh_failed_logins(host: str, attempts: int = 5, timeout: float = 1.0,
                      username: str = "fakeuser", password: str = "WrongPassword123!") -> list[str]:
    out = []
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    for i in range(attempts):
        try:
            client.connect(
                hostname=host,
                port=22,
                username=username,
                password=password,
                timeout=timeout,
                banner_timeout=timeout,
                auth_timeout=timeout,
                allow_agent=False,
                look_for_keys=False,
            )
            out.append(f"[{i+1}/{attempts}] Connexion réussie (inhabituel)" )
        except Exception as e:
            out.append(f"[{i+1}/{attempts}] Échec attendu: {type(e).__name__}")
        time.sleep(0.2)
    try:
        client.close()
    except Exception:
        pass
    return out

## 7. Simulation reconnaissance : scan TCP léger (type "port check")

### Rôle
Tester une liste de ports TCP sur la victime (22, 80, etc.).

### Méthode
On utilise `socket.connect_ex((host, port))` :
- retourne 0 si la connexion TCP est acceptée (port ouvert)
- retourne un code d’erreur sinon (port fermé/filtré)

### Détection attendue
- Snort : alerte scan/reconnaissance selon la règle et le seuil

Ce scan est volontairement simple et pédagogique.

In [ ]:
def mini_tcp_scan(host: str, ports: list[int], timeout: float = 0.5) -> list[str]:
    out = []
    for port in ports:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        start = time.time()
        try:
            code = s.connect_ex((host, port))
            elapsed = (time.time() - start) * 1000
            if code == 0:
                out.append(f"Port {port}: OPEN ({elapsed:.1f} ms)")
            else:
                out.append(f"Port {port}: CLOSED/FILTERED ({elapsed:.1f} ms)")
        except Exception as e:
            out.append(f"Port {port}: ERROR {type(e).__name__}: {e}")
        finally:
            s.close()
        time.sleep(0.03)
    return out

## 8. Exemple d’utilisation (à exécuter sur la machine Attaquant)

Les exemples ci-dessous illustrent comment appeler les fonctions.

### ICMP
`icmp_ping_burst('10.10.10.129', count=20, interval=0.1)`

### HTTP
`http_burst('http://10.10.10.129/', repeat=30)`

### SSH
`ssh_failed_logins('10.10.10.129', attempts=10)`

### Scan
`mini_tcp_scan('10.10.10.129', [22, 80, 443])`

Remarque : dans le cadre du projet, ces appels sont intégrés dans l’application (interface web), mais l’analyse présentée ici concerne uniquement le comportement Python.

## 9. Validation côté SOC (où observer)

### Sur la machine SOC (Snort)
```bash
sudo tail -f /var/log/snort/alert
```

### Sur la machine Victime (logs)
- Nginx :
```bash
sudo tail -f /var/log/nginx/access.log
```
- SSH :
```bash
sudo tail -f /var/log/auth.log
```

### Sur Wazuh Dashboard
Filtrer par l’agent de la victime et vérifier l’apparition d’événements liés à `sshd` et/ou `nginx`.

## 10. Conclusion

Le code Python de simulation génère du trafic réseau et des événements observables à deux niveaux :

- Réseau (Snort) : ICMP, HTTP, tentatives TCP/scan.
- Hôte (Wazuh) : logs `sshd` et logs `nginx` produits sur la machine victime.

Ce script constitue une base contrôlée pour valider la détection avant la phase de corrélation et l’ajout de contre-mesures actives.